# DGD-TabPA — Full Experiments (Google Colab)

End-to-end evaluation pipeline for:

**DGD-TabPA: Diffusion-Guided Dataset Distillation for Tabular Data with Privacy-Aware Evaluation**

### Before you start
1. **Runtime → Change runtime type → GPU** (T4 for draft, A100 for full evaluation runs)
2. Edit **Section 0** knobs
3. Run all cells top-to-bottom (`Runtime → Run all`)

### What this notebook does
1. Clone or mount the project
2. Install dependencies
3. Download datasets
4. Run one DGD-TabPA experiment
5. Run SMOTE baseline
6. Optionally run all ten datasets
7. Optionally run ablation / privacy / repeated-seed suites
8. Build evaluation summary tables + evaluation report
9. Validate outputs and download a zip

Outputs: `DGD-TabPA/outputs/experiments/`


## 0. Configuration

Edit these knobs before running the rest.

| Profile | Suggested settings |
|---------|--------------------|
| **Draft (T4)** | `EPOCHS=20–30`, `TRAIN_ALL_TEN=True`, heavy flags `False` |
| **Full evaluation** | `EPOCHS=50–100`, enable ablations / privacy / seeds |
| **A100** | Higher epochs + all suites enabled |


In [ ]:
# --- Project source ---
# Option A (default): clone from GitHub
REPO_URL = "https://github.com/suranjaliyanage/DGD-TabPA.git"
REPO_BRANCH = "master"
# If the repo is private, set a GitHub PAT (do not commit secrets):
GITHUB_TOKEN = ""  # e.g. "ghp_..."

# Option B: use a Drive folder already uploaded
USE_DRIVE = False
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/DGD-TabPA"

# --- Core experiment knobs ---
PRIMARY_DATASET = "diabetes"
EPOCHS = 30                 # draft: 20-30 | stronger: 50-100+
DISTILL_EPOCHS = 50         # draft: 50 | stronger: 100+
BATCH_SUITE_EPOCHS = 30
DOWNLOAD_DATASETS = "all"   # "all" or e.g. "diabetes"

# Suites
TRAIN_ALL_TEN = True        # all 10 benchmark datasets (DGD)
RUN_BASELINES = True        # SMOTE for primary dataset
RUN_HEAVY = False           # legacy: ablations + privacy + smote suite together
RUN_ABLATIONS = False
RUN_PRIVACY_SWEEP = False
RUN_REPEATED_SEEDS = False
FORCE_RERUN = False         # True = ignore completed runs / status.json

# Reporting
GENERATE_EVALUATION_REPORT = True
REPRESENTATIVE_DATASETS = [
    "diabetes",
    "adult",
    "california_housing",
    "covertype",
]

print("Config ready.")
print(f"PRIMARY_DATASET={PRIMARY_DATASET}, EPOCHS={EPOCHS}, DISTILL_EPOCHS={DISTILL_EPOCHS}")
print(f"TRAIN_ALL_TEN={TRAIN_ALL_TEN}, RUN_BASELINES={RUN_BASELINES}, RUN_HEAVY={RUN_HEAVY}")


## 1. Environment check (GPU)


In [ ]:
import sys
import os
import shutil
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}")
else:
    print("WARNING: No GPU detected. Enable Runtime → Change runtime type → GPU.")


## 2. Get the project code

Clones into `/content/DGD-TabPA` (or mounts Drive if `USE_DRIVE=True`).


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
    if not PROJECT_ROOT.exists():
        raise FileNotFoundError(
            f"Drive path not found: {PROJECT_ROOT}\n"
            "Upload/unzip the project there, or set USE_DRIVE=False to clone from GitHub."
        )
else:
    PROJECT_ROOT = Path("/content/DGD-TabPA")
    if PROJECT_ROOT.exists():
        print(f"Removing existing clone at {PROJECT_ROOT}")
        shutil.rmtree(PROJECT_ROOT)

    clone_url = REPO_URL
    if GITHUB_TOKEN.strip():
        clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN.strip()}@")

    cmd = [
        "git", "clone", "--branch", REPO_BRANCH, "--depth", "1",
        clone_url, str(PROJECT_ROOT),
    ]
    printable = " ".join(cmd)
    if GITHUB_TOKEN:
        printable = printable.replace(GITHUB_TOKEN, "***")
    print(printable)
    subprocess.check_call(cmd)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")
print("Top-level:", sorted(p.name for p in PROJECT_ROOT.iterdir())[:25])


## 3. Install dependencies

Colab already includes PyTorch; install the remaining project requirements.


In [ ]:
!pip -q install -r requirements.txt

import json
import yaml
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

print("Dependencies OK")


In [ ]:
EXPERIMENTS_DIR = PROJECT_ROOT / "outputs" / "experiments"
MASTER_CSV = EXPERIMENTS_DIR / "results_master.csv"
REPORT_DIR = EXPERIMENTS_DIR / "report"
TABLES_DIR = EXPERIMENTS_DIR / "tables"
CONFIG = "config/default.yaml"

def run_cmd(args, check=True):
    cmd = [sys.executable, *args]
    print("\n>>>", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result.returncode

def force_flag():
    return ["--force"] if FORCE_RERUN else []

def show_figures(run_id, max_figs=12):
    fig_dir = EXPERIMENTS_DIR / run_id / "figures"
    if not fig_dir.exists():
        print(f"No figures at {fig_dir}")
        return
    paths = sorted(fig_dir.glob("*.png"))
    print(f"{len(paths)} figures in {fig_dir}")
    for p in paths[:max_figs]:
        display(Markdown(f"**{p.name}**"))
        display(Image(filename=str(p)))

def load_summary(run_id):
    path = EXPERIMENTS_DIR / run_id / "summary_row.json"
    if not path.exists():
        print(f"Missing {path}")
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

print("Helpers ready.")
print(f"EXPERIMENTS_DIR = {EXPERIMENTS_DIR}")


## 4. Download datasets


In [ ]:
if DOWNLOAD_DATASETS == "all":
    run_cmd(["scripts/download_data.py"])
else:
    run_cmd(["scripts/download_data.py", "--dataset", DOWNLOAD_DATASETS])

raw = sorted((PROJECT_ROOT / "data" / "raw").glob("*.csv"))
print(f"Downloaded {len(raw)} CSVs:")
for p in raw:
    print(" -", p.name, f"({p.stat().st_size / 1024:.1f} KB)")


## 5. Single end-to-end DGD-TabPA run

Train → distill → evaluate → figures for `PRIMARY_DATASET`.


In [ ]:
DGD_RUN_ID = f"{PRIMARY_DATASET}_dgd"

run_cmd([
    "scripts/run_experiment.py",
    "--config", CONFIG,
    "--dataset", PRIMARY_DATASET,
    "--method", "dgd_tabpa",
    "--epochs", str(EPOCHS),
    "--distill-epochs", str(DISTILL_EPOCHS),
    "--run-id", DGD_RUN_ID,
    *force_flag(),
])

summary = load_summary(DGD_RUN_ID)
if summary:
    display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
show_figures(DGD_RUN_ID)


## 6. SMOTE baseline (same split / seed / metrics protocol)


In [ ]:
if RUN_BASELINES:
    SMOTE_RUN_ID = f"{PRIMARY_DATASET}_smote"
    run_cmd([
        "scripts/run_experiment.py",
        "--config", CONFIG,
        "--dataset", PRIMARY_DATASET,
        "--method", "smote",
        "--run-id", SMOTE_RUN_ID,
        *force_flag(),
    ])
    summary = load_summary(SMOTE_RUN_ID)
    if summary:
        display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
    show_figures(SMOTE_RUN_ID)
else:
    print("Skipped SMOTE (set RUN_BASELINES=True)")


## 7. All ten datasets (DGD-TabPA)

Runs DGD-TabPA on all 10 benchmarks with a live progress panel:

- current dataset
- percent complete
- elapsed time and ETA remaining

Failed datasets do **not** stop the whole batch. Completed runs are skipped unless `FORCE_RERUN=True`.



In [ ]:
import time
from IPython.display import display, HTML

ALL_TEN = [
    "adult", "churn", "credit", "covertype", "cvd",
    "hcv", "ilpd", "diabetes", "california_housing", "king_county",
]

def _fmt_secs(seconds):
    seconds = max(0, int(round(seconds)))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m {s:02d}s"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"

def show_all_ten_progress(i, total, dataset, run_times, suite_t0, status="running"):
    done = max(i - 1, 0) if status == "running" else i
    pct = 100.0 * done / total if total else 0.0
    remaining_count = total - done
    if run_times:
        avg = sum(run_times) / len(run_times)
        eta_txt = _fmt_secs(avg * remaining_count)
        avg_txt = _fmt_secs(avg)
    else:
        eta_txt = "estimating after first run..."
        avg_txt = "n/a"
    elapsed = _fmt_secs(time.perf_counter() - suite_t0)
    filled = int(30 * done / total) if total else 0
    bar = "#" * filled + "-" * (30 - filled)

    # Visible in Colab logs without wiping training output
    print("\n" + "=" * 72)
    print(f"ALL-TEN PROGRESS  [{bar}] {pct:5.1f}%")
    print(f"Current dataset: {dataset}   | status: {status}   | run: {i}/{total}")
    print(f"Run id: {dataset}_dgd")
    print(f"Elapsed: {elapsed}   | Avg/run: {avg_txt}   | ETA remaining: {eta_txt}")
    print("=" * 72 + "\n", flush=True)

    display(HTML(
        "<div style='font-family:monospace;border:1px solid #888;padding:10px;"
        "border-radius:8px;background:#111;color:#eee'>"
        "<b>All-ten progress</b><br/>"
        f"[{bar}] <b>{pct:.1f}%</b> ({done}/{total} finished)<br/>"
        f"Current: <b>{dataset}</b> ({status}) &nbsp;|&nbsp; {i}/{total}<br/>"
        f"Elapsed: <b>{elapsed}</b> &nbsp;|&nbsp; Avg/run: <b>{avg_txt}</b> "
        f"&nbsp;|&nbsp; ETA: <b>{eta_txt}</b>"
        "</div>"
    ))

if TRAIN_ALL_TEN:
    run_times = []
    suite_t0 = time.perf_counter()
    results = []
    total = len(ALL_TEN)

    for i, ds in enumerate(ALL_TEN, start=1):
        show_all_ten_progress(i, total, ds, run_times, suite_t0, status="running")
        cmd = [
            "scripts/run_experiment.py",
            "--config", CONFIG,
            "--dataset", ds,
            "--method", "dgd_tabpa",
            "--epochs", str(BATCH_SUITE_EPOCHS),
            "--distill-epochs", str(DISTILL_EPOCHS),
            "--run-id", f"{ds}_dgd",
            *force_flag(),
        ]
        t0 = time.perf_counter()
        code = run_cmd(cmd, check=False)
        dt = time.perf_counter() - t0
        run_times.append(dt)
        results.append({
            "dataset": ds,
            "run_id": f"{ds}_dgd",
            "exit_code": code,
            "seconds": round(dt, 1),
        })
        show_all_ten_progress(
            i, total, ds, run_times, suite_t0,
            status="finished" if code == 0 else "failed",
        )
        remaining = total - i
        if remaining and run_times:
            print(
                f"Finished {ds}_dgd in {_fmt_secs(dt)} "
                f"[{i}/{total} = {100.0 * i / total:.1f}%]  "
                f"| ETA remaining: {_fmt_secs((sum(run_times)/len(run_times)) * remaining)}",
                flush=True,
            )
        else:
            print(
                f"Finished {ds}_dgd in {_fmt_secs(dt)} "
                f"[{i}/{total} = {100.0 * i / total:.1f}%]",
                flush=True,
            )

    total_elapsed = _fmt_secs(time.perf_counter() - suite_t0)
    ok = sum(1 for r in results if r["exit_code"] == 0)
    fail = len(results) - ok
    display(HTML(
        "<div style='font-family:monospace;border:1px solid #2a7;padding:10px;"
        "border-radius:8px;background:#e8ffe8;color:#041'>"
        "<b>All-ten complete</b><br/>"
        f"Successful: <b>{ok}</b> &nbsp;|&nbsp; Failed: <b>{fail}</b><br/>"
        f"Total elapsed: <b>{total_elapsed}</b>"
        "</div>"
    ))
    display(pd.DataFrame(results))
else:
    print("Skipped all_ten (set TRAIN_ALL_TEN=True)")

if MASTER_CSV.exists():
    df = pd.read_csv(MASTER_CSV)
    print(f"Master rows: {len(df)}")
    cols = [
        c for c in [
            "run_id", "dataset", "method", "status",
            "mean_tstr_f1", "mean_tstr_auc", "mean_tstr_r2",
            "dcr_median", "total_runtime_seconds",
        ] if c in df.columns
    ]
    display(df[cols].tail(20))
else:
    print("No results_master.csv yet")



## 8. Optional heavy suites (legacy `RUN_HEAVY`)

When `RUN_HEAVY=True`, also runs ablations, privacy sweep, and SMOTE across datasets.
Prefer the finer flags in Sections 13–15 for full evaluation runs.


In [ ]:
if RUN_HEAVY:
    # SMOTE across datasets
    cmd = [
        "scripts/run_batch_experiments.py",
        "--suite", "smote",
        "--epochs", str(BATCH_SUITE_EPOCHS),
    ]
    if FORCE_RERUN:
        cmd.append("--force")
    run_cmd(cmd, check=False)

    for suite in ("ablations", "privacy"):
        cmd = [
            "scripts/run_batch_experiments.py",
            "--suite", suite,
            "--dataset", PRIMARY_DATASET,
            "--epochs", str(BATCH_SUITE_EPOCHS),
            "--distill-epochs", str(DISTILL_EPOCHS),
        ]
        if FORCE_RERUN:
            cmd.append("--force")
        run_cmd(cmd, check=False)
else:
    print("Heavy suites skipped (set RUN_HEAVY=True, or use Sections 13-15 flags).")


## 9. Collect master results table


In [ ]:
if MASTER_CSV.exists():
    master = pd.read_csv(MASTER_CSV)
    print(f"Loaded {len(master)} rows from {MASTER_CSV}")
    display(master.head(30))

    key_cols = [
        "run_id", "dataset", "method", "ablation", "task_type", "status",
        "mean_tstr_f1", "mean_tstr_auc", "mean_tstr_r2", "rmse",
        "wasserstein_mean", "pcd", "dcr_median", "exact_copy_count", "mia_auc",
        "epsilon", "total_runtime_seconds",
    ]
    present = [c for c in key_cols if c in master.columns]
    out_table = EXPERIMENTS_DIR / "results_table.csv"
    master[present].to_csv(out_table, index=False)
    print(f"Wrote compact table: {out_table}")
    display(master[present])
else:
    print("results_master.csv not found — run experiments first.")


## 10. Display generated figures (primary runs)


In [ ]:
for run_id in [f"{PRIMARY_DATASET}_dgd", f"{PRIMARY_DATASET}_smote"]:
    display(Markdown(f"### {run_id}"))
    show_figures(run_id, max_figs=8)

if REPORT_DIR.exists():
    display(Markdown("### Aggregated evaluation figures"))
    for p in sorted(REPORT_DIR.glob("*.png"))[:12]:
        display(Markdown(f"**{p.name}**"))
        display(Image(filename=str(p)))


## 11. Generate evaluation summary tables

Builds `outputs/experiments/tables/table_*.csv` and related artefacts.


In [ ]:
if GENERATE_EVALUATION_REPORT:
    run_cmd(["scripts/build_evaluation_report.py", "--dataset", PRIMARY_DATASET])
    if TABLES_DIR.exists():
        print("Tables:")
        for p in sorted(TABLES_DIR.glob("*")):
            print(" -", p.name)
else:
    print("Skipped (set GENERATE_EVALUATION_REPORT=True)")


## 12. Generate aggregated evaluation figures

Produced by `build_evaluation_report.py` (comparison / privacy / ablation / paired plots).


In [ ]:
if REPORT_DIR.exists():
    figs = sorted(REPORT_DIR.glob("*.png"))
    print(f"{len(figs)} aggregated figures in {REPORT_DIR}")
    for p in figs:
        print(" -", p.name)
    for p in figs[:10]:
        display(Markdown(f"**{p.name}**"))
        display(Image(filename=str(p)))
else:
    print("No report folder yet — run Section 11 first.")


## 13. Run repeated-seed experiments

Seeds come from `config/default.yaml` (`experiments.seeds`).


In [ ]:
if RUN_REPEATED_SEEDS:
    cmd = [
        "scripts/run_batch_experiments.py",
        "--suite", "seeds",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
        "--distill-epochs", str(DISTILL_EPOCHS),
    ]
    run_cmd(cmd, check=False)
    if GENERATE_EVALUATION_REPORT:
        run_cmd(["scripts/build_evaluation_report.py", "--dataset", PRIMARY_DATASET])
else:
    print("Skipped repeated seeds (set RUN_REPEATED_SEEDS=True)")


## 14. Run privacy–utility sweep

Epsilon grid from YAML (default includes 0.5, 1, 2, 4, 8, 16, 100).


In [ ]:
if RUN_PRIVACY_SWEEP:
    cmd = [
        "scripts/run_batch_experiments.py",
        "--suite", "privacy",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
        "--distill-epochs", str(DISTILL_EPOCHS),
    ]
    if FORCE_RERUN:
        cmd.append("--force")
    run_cmd(cmd, check=False)
    if GENERATE_EVALUATION_REPORT:
        run_cmd(["scripts/build_evaluation_report.py", "--dataset", PRIMARY_DATASET])
else:
    print("Skipped privacy sweep (set RUN_PRIVACY_SWEEP=True)")


## 15. Run ablation suite

Supported: `mlp_denoiser`, `no_attention`, `minmax`, `raw_space`.


In [ ]:
if RUN_ABLATIONS:
    cmd = [
        "scripts/run_batch_experiments.py",
        "--suite", "ablations",
        "--dataset", PRIMARY_DATASET,
        "--epochs", str(BATCH_SUITE_EPOCHS),
        "--distill-epochs", str(DISTILL_EPOCHS),
    ]
    if FORCE_RERUN:
        cmd.append("--force")
    run_cmd(cmd, check=False)
    if GENERATE_EVALUATION_REPORT:
        run_cmd(["scripts/build_evaluation_report.py", "--dataset", PRIMARY_DATASET])
else:
    print("Skipped ablations (set RUN_ABLATIONS=True)")


## 16. Generate evaluation Markdown report

Evidence-only report + figure captions under `outputs/experiments/report/`.


In [ ]:
if GENERATE_EVALUATION_REPORT:
    run_cmd(["scripts/build_evaluation_report.py", "--dataset", PRIMARY_DATASET])
    report = REPORT_DIR / "evaluation_report.md"
    captions = REPORT_DIR / "figure_captions.md"
    if report.exists():
        display(Markdown(report.read_text(encoding="utf-8")[:8000]))
    if captions.exists():
        print(f"Captions: {captions}")
else:
    print("Skipped report generation")


## 17. Validate output completeness


In [ ]:
rc = run_cmd(["scripts/validate_experiment_outputs.py"], check=False)
print("Validation exit code:", rc, "(0 = no critical errors)")

# After a full all_ten run you can also require all datasets:
# run_cmd(["scripts/validate_experiment_outputs.py", "--require-all-ten"], check=False)

if MASTER_CSV.exists():
    df = pd.read_csv(MASTER_CSV)
    if "dataset" in df.columns:
        print("Datasets present:", sorted(df["dataset"].dropna().unique().tolist()))
    if "method" in df.columns:
        print("Methods present:", sorted(df["method"].dropna().unique().tolist()))


## 18. Package outputs for download


In [ ]:
zip_base = Path("/content/DGD-TabPA_experiments")
zip_file = zip_base.with_suffix(".zip")
if zip_file.exists():
    zip_file.unlink()

zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(EXPERIMENTS_DIR)))
print(f"Created: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as e:
        print("Auto-download failed:", e)
        print("Download manually from the Colab file browser:", zip_path)
else:
    print("Not in Colab — zip left at", zip_path)


## 19. Optional: copy outputs to Google Drive


In [ ]:
SAVE_TO_DRIVE = False  # set True to persist results
DRIVE_OUT = "/content/drive/MyDrive/DGD-TabPA_outputs"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    target = Path(DRIVE_OUT)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(EXPERIMENTS_DIR, target)
    print(f"Copied experiments to {target}")
else:
    print("Drive copy skipped (set SAVE_TO_DRIVE=True)")


## Notes

- Resume: completed runs with `status.json` + `summary_row.json` are skipped unless `FORCE_RERUN=True`.
- Do not interpret a single p-value as proof of superiority.
- Unavailable baselines (TVAE, CTAB-GAN+, TabDDPM, MTabGen) are registered but **not fabricated**.
- For stronger numbers, raise `EPOCHS` / `DISTILL_EPOCHS` and enable privacy / ablation / seed suites.
